## Merge Kecamatan
menggabungkan semua data kecamatan dalam 1 kota yang sama

In [5]:
import os
import json
import re
import csv
    
INPUT_FOLDER = "./denpasar"
CSV_FILE = "./denpasar/denpasar.csv"
OUTPUT_FILE = "./map_kota/map_denpasar.geojson"

# ==========================================================
# LOAD DATA PENDUDUK DARI CSV
# ==========================================================

population_data = {}

if os.path.exists(CSV_FILE):
    with open(CSV_FILE, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter=";")

        for row in reader:
            kode = row["Kode Kecamatan"].strip()

            population_data[kode] = {
                # "pria": int(row["Laki-Laki"].replace(".", "")),
                "pria": int(row[[k for k in row.keys() if k.lower() == "laki-laki"][0]].replace(".", "")),
                "wanita": int(row["Perempuan"].replace(".", "")),
                "total_penduduk": int(row["Jumlah"].replace(".", ""))
            }

    print(f"Total data penduduk kecamatan: {len(population_data)} kecamatan")

else:
    print(f"CSV '{CSV_FILE}' tidak ditemukan")


# ==========================================================
# GEOJSON TEMPLATE
# ==========================================================

merged_geojson = {
    "type": "FeatureCollection",
    "name": "map",
    "crs": {
        "type": "name",
        "properties": {
            "name": "urn:ogc:def:crs:OGC:1.3:CRS84"
        }
    },
    "features": []
}

if not os.path.exists(INPUT_FOLDER):
    print(f"Folder '{INPUT_FOLDER}' tidak ditemukan!")
    exit()

files = [f for f in os.listdir(INPUT_FOLDER) if f.endswith('.geojson') or f.endswith('.json')]
print(f"Memproses {len(files)} file kecamatan...")

for filename in files:
    filepath = os.path.join(INPUT_FOLDER, filename)
    
    clean_filename = os.path.splitext(filename)[0]
    
    
    # ------------------------------------------------------
    # FORMAT:
    # id3578010_Karang_Pilang
    # atau
    # 3578010_Karang_Pilang
    # ------------------------------------------------------
    # ekstrak kode kecamatan dari nama file geojson
    match = re.search(r"(?:id)?(\d+)", clean_filename, re.IGNORECASE)

    if match:
        district_id = match.group(1)
    else:
        district_id = None


    # ekstrak nama daerah dr file geojson
    raw_name = re.sub(
        r"^(?:id)?\d+[_\-\s]*",
        "",
        clean_filename,
        flags=re.IGNORECASE
    )

    district_name = (
        raw_name
        .replace("_", " ")
        .replace("-", " ")
        .strip()
        .title()
    )


    with open(filepath, 'r', encoding='utf-8') as f:
        try:
            data = json.load(f)
            
            extracted_geometry = None
            if data.get("type") == "GeometryCollection" and "geometries" in data:
                extracted_geometry = data["geometries"][0]
            elif data.get("type") == "FeatureCollection" and "features" in data:
                extracted_geometry = data["features"][0].get("geometry")
            elif data.get("type") == "Feature":
                extracted_geometry = data.get("geometry")
            elif data.get("type") in ["Polygon", "MultiPolygon"]:
                extracted_geometry = data

            if not extracted_geometry:
                print(f"⚠️ Gagal mengekstrak geometri: {filename}")
                continue

            # --------------------------------------------------
            # AMBIL DATA PENDUDUK BERDASARKAN KODE KECAMATAN
            # --------------------------------------------------

            penduduk = population_data.get(
                district_id,
                {
                    "pria": 0,
                    "wanita": 0,
                    "total_penduduk": 0
                }
            )

            properties = {
                "district_id": district_id,
                "province": "Bali",
                "regency": "Denpasar",
                "district": district_name,
                "pria": penduduk["pria"],
                "wanita": penduduk["wanita"],
                "total_penduduk": penduduk["total_penduduk"]
            }

            feature = {
                "type": "Feature",
                "properties": properties,
                "geometry": extracted_geometry
            }

            merged_geojson["features"].append(feature)
            print(
                f"[{district_id}] {district_name}"
                f"| Penduduk: {penduduk['total_penduduk']:,}"
            )

        except Exception as e:
            print(f"Error pada file {filename}: {e}")

# ------------------------------------------------------------------
# POST-PROCESSING UNTUK FORMAT DATA COORDINATES
# ------------------------------------------------------------------
raw_json = json.dumps(merged_geojson, ensure_ascii=False, indent=2)

# Fungsi untuk membuang newline & spasi berlebih di dalam blok "coordinates": [...]
def collapse_coordinates(match):
    coord_str = match.group(0)

    # Pisahkan bagian penutup "}"
    body = coord_str[:-1]
    closing = "}"

    body = re.sub(r'\s+', ' ', body)
    body = re.sub(r'\[\s+', '[', body)
    body = re.sub(r'\s+\]', ']', body)
    body = re.sub(r',\s*', ', ', body)

    return body + "\n      " + closing

# pattern "coordinates": [ ... ] lalu buat satu baris menyamping
compact_json = re.sub(
    r'"coordinates":\s*\[[\s\S]*?\]\s*\}',
    collapse_coordinates,
    raw_json
)

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    f.write(compact_json)

print(f"\nBerhasil! Seluruh isi koordinat disatukan menyamping di '{OUTPUT_FILE}'.")

CSV './denpasar/denpasar.csv' tidak ditemukan
Memproses 5 file kecamatan...
[5171010] Denpasar Selatan| Penduduk: 0
[5171020] Denpasar Timur| Penduduk: 0
[5171030] Denpasar Barat| Penduduk: 0
[5171031] Denpasar Utara| Penduduk: 0
[5171] Kota Denpasar| Penduduk: 0

Berhasil! Seluruh isi koordinat disatukan menyamping di './map_kota/map_denpasar.geojson'.


## Merge Kota
menggabungkan semua data tiap kota menjadi 1 map.geojson

In [6]:
import os
import json
import re

INPUT_FOLDER = "./map_kota"
OUTPUT_FILE = "./indonesia.geojson"

merged = {
    "type": "FeatureCollection",
    "name": "map",
    "crs": {
        "type": "name",
        "properties": {
            "name": "urn:ogc:def:crs:OGC:1.3:CRS84"
        }
    },
    "features": []
}

for filename in os.listdir(INPUT_FOLDER):
    if not filename.endswith(".geojson"):
        continue

    filepath = os.path.join(INPUT_FOLDER, filename)

    try:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)

        if data.get("type") != "FeatureCollection":
            print(f"⚠️ {filename} bukan FeatureCollection")
            continue

        feature_count = len(data.get("features", []))
        merged["features"].extend(data.get("features", []))

        print(f"{filename} -> {feature_count} feature")

    except Exception as e:
        print(f"Gagal memproses {filename}: {e}")

print("\n=====================================")
print(f"Total file diproses : {len([f for f in os.listdir(INPUT_FOLDER) if f.endswith('.geojson')])}")
print(f"Total feature       : {len(merged['features'])}")
print("=====================================")

# ==========================================================
# POST PROCESSING AGAR COORDINATES MENJADI SATU BARIS
# ==========================================================

raw_json = json.dumps(merged, ensure_ascii=False, indent=2)

def collapse_coordinates(match):
    coord_str = match.group(0)

    # Pisahkan bagian penutup "}"
    body = coord_str[:-1]
    closing = "}"

    body = re.sub(r'\s+', ' ', body)
    body = re.sub(r'\[\s+', '[', body)
    body = re.sub(r'\s+\]', ']', body)
    body = re.sub(r',\s*', ', ', body)

    return body + "\n      " + closing

compact_json = re.sub(
    r'"coordinates":\s*\[[\s\S]*?\]\s*\}',
    collapse_coordinates,
    raw_json
)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(compact_json)

print(f"Berhasil menggabungkan {len(merged['features'])} feature.")

map_denpasar.geojson -> 5 feature
map_gresik.geojson -> 19 feature
map_jakarta_barat.geojson -> 9 feature
map_jakarta_pusat.geojson -> 9 feature
map_jakarta_selatan.geojson -> 11 feature
map_jakarta_timur.geojson -> 11 feature
map_jakarta_utara.geojson -> 9 feature
map_kota_bandung.geojson -> 31 feature
map_makassar.geojson -> 16 feature
map_sidoarjo.geojson -> 18 feature
map_surabaya.geojson -> 32 feature

Total file diproses : 11
Total feature       : 170
Berhasil menggabungkan 170 feature.
